# VolCast: exploratory data analysis

What the data looks like, and how it shaped the design. Each section links a pattern in the data to a decision in `docs/hld.md`.

Run after the pipeline has built `data/features/features.parquet` (`dvc repro features`).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "dvc.yaml").exists())
df = pd.read_parquet(ROOT / "data/features/features.parquet").sort_values(["symbol", "open_time"])
df["time"] = pd.to_datetime(df["open_time"], unit="ms", utc=True)
plt.rcParams.update({"figure.figsize": (11, 4), "axes.grid": True, "grid.alpha": 0.3})
COINS = ["BTCUSDT", "ETHUSDT", "SOLUSDT"]

df.groupby("symbol").agg(rows=("target", "size"), start=("time", "min"), end=("time", "max"),
                         median_next_hour_vol_pct=("target", lambda s: round(100 * s.median(), 3)))

One row per coin per minute over the training window. `target` is the realized volatility of the next 60 minutes, the number the model forecasts.

## 1. Volatility comes in clusters

Calm and wild stretches each last hours to days. HLD 4.3: recent volatility is the strongest predictor of the next hour's.

In [ ]:
fig, ax = plt.subplots()
for coin in COINS:
    s = df[df["symbol"] == coin].set_index("time")["rv_1h"].resample("1h").last() * 100
    ax.plot(s.index, s, lw=0.8, label=coin)
ax.set(title="Past-hour realized volatility, sampled hourly", ylabel="% move per hour")
ax.legend();

## 2. Volatility depends on the time of day and week

Activity peaks when US and European markets overlap and drops at weekends. HLD 4.3: the `hour` and `weekday` features.

In [ ]:
btc = df[df["symbol"] == "BTCUSDT"]
grid = btc.pivot_table(index="weekday", columns="hour", values="target", aggfunc="median") * 100
fig, ax = plt.subplots(figsize=(11, 3.5))
im = ax.imshow(grid, aspect="auto", cmap="viridis")
ax.set(title="BTC median next-hour volatility (%) by weekday and hour (UTC)", xlabel="Hour (UTC)",
       yticks=range(7), yticklabels=["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"])
ax.grid(False)
fig.colorbar(im, ax=ax);

## 3. The target is skewed with a heavy right tail

Most hours are quiet, a few are extreme. On a log scale it looks close to normal. HLD 4.4 and 4.5: QLIKE and the gamma objective handle this scale better than squared error, which large hours would dominate.

In [ ]:
fig, (a, b) = plt.subplots(1, 2)
y = df["target"] * 100
a.hist(y, bins=200, range=(0, y.quantile(0.995)))
a.set(title="Next-hour volatility (%)", xlabel="% move")
b.hist(np.log(df["target"]), bins=200)
b.set(title="log of next-hour volatility", xlabel="log")
print(f"skewness: raw {y.skew():.2f}, log {np.log(df['target']).skew():.2f}")

## 4. Features are strongly autocorrelated

Consecutive minutes share almost the whole rolling window, so a feature barely changes from one minute to the next. HLD 4.10 and experiment E4: tests that assume independent rows (like PSI) break on this data, so drift is measured against training percentiles instead. It is also why the live-error alert waits 6 hours.

In [ ]:
lags = [1, 5, 15, 60, 240, 1440]
acf = pd.DataFrame({f: [btc[f].autocorr(lag) for lag in lags] for f in ["rv_15m", "rv_1h", "rv_24h", "target"]},
                   index=[f"{lag} min" for lag in lags])
acf.round(3)

A value near 1 means the feature an hour later is almost the same number. `rv_24h` stays near 1 even a full day apart.

## 5. Past volatility predicts the next hour, with noise

The baselines rely on this link: persistence uses `rv_1h` directly, HAR combines it with `rv_4h` and `rv_24h`. The spread around the diagonal is what LightGBM tries to explain with the other features.

In [ ]:
sample = df.sample(20_000, random_state=0)
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(sample["rv_1h"] * 100, sample["target"] * 100, s=2, alpha=0.3)
ax.set(xscale="log", yscale="log", xlabel="Past-hour volatility (%)", ylabel="Next-hour volatility (%)",
       title="Past hour vs next hour")
lim = ax.get_xlim()
ax.plot(lim, lim, color="red", lw=1, label="next = past (persistence)")
ax.legend()
print(f"correlation of logs: {np.log(df['rv_1h']).corr(np.log(df['target'])):.2f}")

## 6. Coins move by different amounts

SOL typically moves about twice as much as BTC. HLD 4.4: `symbol_id` is a feature, so one model can serve all three coins.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.boxplot([df.loc[df["symbol"] == c, "target"] * 100 for c in COINS], showfliers=False)
ax.set(xticks=[1, 2, 3], xticklabels=COINS, ylabel="Next-hour volatility (%)", title="Typical hourly move by coin")
(df.groupby("symbol")["target"].median() * 100).round(3).rename("median %")

## Summary

| Finding | Design decision |
|---|---|
| Volatility clusters | recent-volatility features over several windows |
| Daily and weekly patterns | `hour` and `weekday` features |
| Skewed, heavy-tailed target | QLIKE for selection, gamma objective for training |
| Strong autocorrelation | percentile drift checks instead of PSI; sustained alert windows |
| Past predicts next, noisily | persistence and HAR baselines, LightGBM for the rest |
| Coins differ in scale | `symbol_id` feature, one model for all coins |